<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_5_Tool_Use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 — Tool Use: How AI Calls Functions


### 👋 Who this session is for

You. Right now, you only need to know two things:

1. An **LLM** (Large Language Model) is an AI, like Claude, that reads text and writes text.
2. A **prompt** is the message you send to it.

That's enough. Everything else is explained from zero.

---

### 🎯 What you'll be able to do by the end

- **Explain** what tool use is, in one minute, in your own words.
- **Explain** how the AI *decides* to call a tool.
- **Draw and explain** the tool call loop: AI → tool → result → AI continues.
- **Build** a working tool-using assistant with the real Claude API.
- **Name** the built-in tools Claude itself uses (web search, file reading, code execution) and explain how they work.

---

### 📐 How each section is structured

| Block | What it gives you |
|---|---|
| The simple idea | Plain-English explanation with an analogy |
| Real code | A short cell you actually run against the live Claude API |
| Remember This | One line you can say in an interview |
| Don't mix these up | Common wrong beliefs, corrected |
| Quick check | A tiny question (answer given) to test yourself |

---

---
## Section 0 — Foundations: Why Does AI Even Need Tools?

### 🧠 Quick recap (the only two things you need)

- An **LLM** learned from a huge amount of text. When you send a prompt, it writes back the most sensible reply it can.
- Its knowledge was **frozen** on the day its training ended. It's like a very well-read person who has been asleep since that day.

### 🚧 Three things an LLM alone CANNOT do

1. **Know live information.** Ask "What is the weather in Chennai right now?" — it cannot know. Its training data is from the past.
2. **Guarantee exact calculations.** It predicts text. It usually gets math right, but "usually" is not good enough for a bank statement.
3. **Take actions.** It cannot send an email, look up your order in a database, or book a ticket. It can only *write text*.

### 💡 The fix

We give the AI **tools** — small pieces of real code it can ask us to run. A tool can check the weather, query a database, do exact math, or send an email.

This one idea — **tool use** (also called **function calling**) — is what turns a chatbot into an assistant that can actually *do things*. It is the foundation of every AI agent you will ever build.

---

### 🧠 Quick check

**True or False:** An LLM can directly check today's gold price.

> **Answer: False.** Its knowledge is frozen at training time. It needs a tool (like web search) to get live data.


---
## Section 1 — What is Tool Use?

### ❓ What is it?

**Tool use** means: we describe some functions to the AI, and the AI can *ask us to run one* when it needs it. We run the function, give the result back, and the AI uses that result to answer the user.

### 🧠 Analogy 1 — The Smart Manager

Imagine a very smart manager at a company. She knows a LOT — strategy, history, language, logic. But she does not do every task with her own hands:

- Someone asks "what time is it in Tokyo?" → she checks her phone.
- Someone needs a 200-row calculation → she opens Excel.

**She is the brain. The tools are her specialists.** She decides *which* tool to use and *when* — the tools just do their one job well.

Claude works the same way. Claude is the brain. Your functions are the specialists.

### 📱 Analogy 2 — The Calculator on Your Phone

When maths gets too hard to do in your head, *your brain* decides: "I'll open the calculator." The calculator doesn't decide anything — it just computes.

- Claude = your brain (decides *when* and *what*)
- Tools = the apps (calculator, clock, search)

### ❓ Why does it matter?

Because this is how real AI products work. A bank's AI assistant answers "what's my balance?" not from memory, but by calling a `get_balance` tool that queries the bank's real database. No tools = no real product.

### 🔧 What exactly IS a tool?

A tool has **two parts**:

| Part | What it is | Who reads/runs it |
|---|---|---|
| **A function** | The real code that does the work | *Your computer* runs it |
| **A schema** | A short description of the function, written as a dictionary | *Claude* reads it |

The **schema** is like a **job description** for the tool. Claude reads it and learns: "What is this tool called? What does it do? What inputs does it need?"

> **Accuracy note:** Claude never sees your function's code and never runs it. Claude only sees the *description*. This matters — it means *you* stay in full control of what actually executes.

Let's look at the simplest possible tool. Run the next cell — it's pure Python, no AI yet.


In [1]:
import datetime

# --- PART 1: THE FUNCTION ---
# This is the real code that runs when the tool is called.
def get_current_time():
    now = datetime.datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

# --- PART 2: THE SCHEMA (the "job description" Claude reads) ---
get_current_time_tool = {
    "name": "get_current_time",
    "description": "Returns the current date and time. Use when the user asks what time or date it is right now.",
    "input_schema": {
        "type": "object",
        "properties": {},   # this tool needs no inputs
        "required": []
    }
}

# Test the function directly (no AI involved yet)
print("Current time:", get_current_time())
print()
print("The schema Claude will read:")
print(get_current_time_tool)

Current time: 2026-07-11 10:03:21

The schema Claude will read:
{'name': 'get_current_time', 'description': 'Returns the current date and time. Use when the user asks what time or date it is right now.', 'input_schema': {'type': 'object', 'properties': {}, 'required': []}}


### 📋 When does the AI use a tool?

Not every question needs a tool. Claude only asks for a tool when it genuinely helps:

| User prompt | Tool used? | Why |
|---|---|---|
| "What is 2 + 2?" | ❌ No | Claude knows this from training |
| "What time is it in Tokyo right now?" | ✅ Yes | Needs live data |
| "Explain photosynthesis" | ❌ No | General knowledge |
| "What is the BMI of someone 170cm, 70kg?" | ✅ Yes | A calculation tool makes it exact and reliable |

> **Accuracy note:** Claude's choice is guided by the tool's **description**. A clear description ("Use this when the user asks about the time") makes Claude pick the right tool. A vague description confuses it. Writing good descriptions is a real, paid skill.

### 🌍 Real-world example

When you ask a food delivery app's AI "Where is my order?", the AI calls a `get_order_status(order_id)` tool that hits the company's live database. The AI writes the friendly sentence; the tool fetches the truth.

### 💬 Remember This

> *"A tool is a function plus a description. The AI reads the description and decides WHEN to call it — but your code is what actually RUNS it."*

### ⚠️ Don't mix these up

- ❌ **Wrong:** "The AI runs my Python function."
  ✅ **Right:** The AI only *requests* the call. Your code runs the function.
- ❌ **Wrong:** "If I give Claude tools, it will always use them."
  ✅ **Right:** Claude uses a tool only when the question needs it.
- ❌ **Wrong:** "Tools let the AI do anything."
  ✅ **Right:** The AI can only use the exact tools *you* define. Nothing more.

### 🎤 Interview question

**Q: "What is tool use (function calling) in one sentence?"**
> *Model answer:* "It's a mechanism where the LLM, instead of answering directly, returns a structured request asking my code to run a specific function with specific inputs — my code runs it, sends the result back, and the LLM writes the final answer."

### 🧠 Quick check

Which of these two prompts needs a tool, and why?
**(a)** "Who wrote Romeo and Juliet?"  **(b)** "What is today's date?"

> **Answer:** (a) No tool — Shakespeare is in training data. (b) Tool — the AI's knowledge is frozen in the past, so it cannot know *today's* date without a tool.


---
## Section 2 — How the AI Decides to Call a Tool

### 🔍 Claude's decision process

When your message arrives, and tools are available, Claude thinks through this:

1. **Read the user's message** — what is being asked?
2. **Read the tool descriptions** — what tools do I have?
3. **Decide** — "Can I answer from my training? Or do I need a tool?"
4. **If no tool needed** → reply with a normal text answer.
5. **If a tool is needed** → reply with a special `tool_use` block: *"Please run this tool, with these inputs."*

🔑 **The most important sentence of this whole session:**

> **Claude NEVER runs your function. YOUR code runs it.**
> Claude only says: "Please call `calculate_bmi` with height=170, weight=68." Then it waits.

### 📝 Example A — AI answers directly (no tool)

```
User: "What does BMI stand for?"

AI: "BMI stands for Body Mass Index..."

→ No tool_use block. Claude answered from training —
  even though a calculate_bmi tool was available.
```

### 📝 Example B — AI requests a tool call

```
User: "What is the BMI of someone 170cm tall and 68kg?"

AI's reply (simplified):
{
  "type": "tool_use",
  "name": "calculate_bmi",
  "input": { "height_cm": 170, "weight_kg": 68 }
}

→ Claude did NOT calculate anything.
→ It picked the right tool AND extracted the inputs (170, 68) from plain English.
→ Now YOUR code must run calculate_bmi(170, 68) and send the result back.
```

Notice something quietly amazing in Example B: the user never typed "height_cm=170". Claude *read the sentence* and mapped it to the schema's inputs by itself. That translation — messy human language → clean structured inputs — is half the magic of tool use.

### 📝 Example C — No suitable tool exists

```
User: "What is the weather in Chennai right now?"
Tools available: [calculate_bmi, convert_currency]

→ Claude will say it cannot check the weather.
  It has no weather tool, and it cannot invent one.
```

**Lesson:** Claude can only use tools YOU provide. Need weather? You must build and offer a weather tool.

### 💬 Remember This

> *"The AI decides WHETHER and WHICH tool to call, and with WHAT inputs. Your code decides whether to actually run it. The intelligence is in the choosing; the power stays with you."*

### 🧠 Quick check

You give Claude a `search_web(query)` tool and ask: *"Who won the IPL in 2024?"* What happens?

> **Answer:** Claude returns a `tool_use` block asking your code to run `search_web(query="IPL 2024 winner")`. Your code runs the search, sends the result back, and Claude writes the final answer using it.


---
## 🔬 Lab Setup — Connect to the Real Claude API

**Objective:** get a working Claude client in 2 cells.

Before running: in Colab, click the 🔑 key icon (left sidebar) → add a secret named `MY_API_KEY` with your Anthropic API key.

> 💰 **Cost note:** we use **Claude Haiku** — the small, fast, cheap model. This whole session costs only a few cents. 🔒 **Safety note:** never paste a real key directly into a notebook cell — always use Colab Secrets.


In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 45.9 MB/s eta 0:00:00


In [3]:
# Key + client (Colab Secret named MY_API_KEY)
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"
print("Ready ✅")

Ready ✅


---
## 🔬 Lab 1 — Watch Claude Decide

**Objective:** send two prompts with the *same* tool available, and see Claude use the tool for one but not the other.

The signal to watch is **`stop_reason`** — Claude's way of telling your code why it stopped writing:

| `stop_reason` | Meaning |
|---|---|
| `"end_turn"` | "I'm done — here is my answer." |
| `"tool_use"` | "I need a tool — please run it and get back to me." |


In [4]:
# One tool available for BOTH prompts
calculate_bmi_tool = {
    "name": "calculate_bmi",
    "description": "Calculates the Body Mass Index (BMI) for a person given their height and weight.",
    "input_schema": {
        "type": "object",
        "properties": {
            "height_cm": {"type": "number", "description": "The person's height in centimetres"},
            "weight_kg": {"type": "number", "description": "The person's weight in kilograms"}
        },
        "required": ["height_cm", "weight_kg"]
    }
}

# --- PROMPT A: Claude can answer from knowledge ---
print("=== PROMPT A: 'What does BMI stand for?' ===")
response_a = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[calculate_bmi_tool],
    messages=[{"role": "user", "content": "What does BMI stand for?"}]
)
print("Stop reason:", response_a.stop_reason)
for block in response_a.content:
    if block.type == "text":
        print("Text:", block.text)

print()
# --- PROMPT B: Claude needs the tool ---
print("=== PROMPT B: 'What is the BMI of someone 170cm and 68kg?' ===")
response_b = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[calculate_bmi_tool],
    messages=[{"role": "user", "content": "What is the BMI of someone who is 170cm tall and weighs 68kg?"}]
)
print("Stop reason:", response_b.stop_reason)
for block in response_b.content:
    print("Block type:", block.type)
    if block.type == "tool_use":
        print("  Tool name:", block.name)
        print("  Tool inputs:", block.input)
        print("  Tool use ID:", block.id)

=== PROMPT A: 'What does BMI stand for?' ===
Stop reason: end_turn
Text: BMI stands for **Body Mass Index**. 

It's a measure of body fat based on height and weight that applies to adult men and women. BMI is calculated by dividing a person's weight in kilograms by the square of their height in meters (kg/m²). 

BMI is commonly used as a screening tool to identify potential weight categories that may lead to health problems, though it's important to note that BMI is just one indicator and doesn't directly measure body fat percentage or overall health.

If you'd like me to calculate someone's BMI, I'd be happy to help! I just need their height (in centimeters) and weight (in kilograms).

=== PROMPT B: 'What is the BMI of someone 170cm and 68kg?' ===
Stop reason: tool_use
Block type: tool_use
  Tool name: calculate_bmi
  Tool inputs: {'height_cm': 170, 'weight_kg': 68}
  Tool use ID: toolu_01ShwYBbTe3QrAgP1pvA9uTb


**Expected result:** Prompt A ends with `stop_reason: end_turn` and a text answer. Prompt B ends with `stop_reason: tool_use` and a request to call `calculate_bmi` with `{'height_cm': 170, 'weight_kg': 68}` — and **no final answer yet**. Claude is waiting for you.

**Try this:** change Prompt B to *"My friend is 5 foot 6 and 60 kilos — BMI?"* Watch Claude convert feet to centimetres by itself before filling the inputs.

### ⚠️ Don't mix these up

- ❌ **Wrong:** "`stop_reason: tool_use` means Claude answered the question."
  ✅ **Right:** It means Claude is *paused, waiting* for a tool result. There is no answer yet.
- ❌ **Wrong:** "The tool ran somewhere when I saw `tool_use`."
  ✅ **Right:** Nothing ran. It's only a *request*. Until your code runs the function, nothing has executed.

### 🎤 Interview question

**Q: "How does your code know whether Claude wants a tool or has finished answering?"**
> *Model answer:* "Check `stop_reason`. If it's `tool_use`, the response contains one or more `tool_use` blocks with the tool name, inputs, and an ID — I run the tools and send results back. If it's `end_turn`, the answer is final."


---
## Section 3 — The Tool Call Loop (The Heart of Today)

### 🔄 The 5-step loop

Every tool call follows this same pattern. Memorise this and you understand 90% of tool use:

```
Step 1: You send  →  the user's message + tool definitions  →  to Claude
                        ↓
Step 2: Claude returns a tool_use block  ("please run this")  — NOT an answer yet
                        ↓
Step 3: YOUR code runs the actual Python function
                        ↓
Step 4: You send the tool RESULT back to Claude
                        ↓
Step 5: Claude reads the result and writes the final answer
```

Think of it as a **relay race**: you pass the baton to Claude → Claude passes back a request → your code runs the function → you pass the result back → Claude crosses the finish line with the answer.

This is exactly what people mean by: **AI → tool → result → AI continues.**

> **Accuracy note:** this loop describes **client tools** — tools *you* run. Anthropic also offers **server tools** (like web search) where Anthropic runs the tool on *their* computers and the loop happens invisibly inside one API call. We meet those in Section 5.

### 🔬 Lab 2 — The full loop, one tool, one call

**Objective:** run the complete 5-step loop end to end for a BMI question. Read every comment — they narrate each step.


In [5]:
# ============================================================
# LAB 2: The complete tool call loop — BMI calculator
# ============================================================

# --- STEP 0: Define the function and its schema ---

# The function — this is what ACTUALLY calculates (your computer runs this)
def calculate_bmi(height_cm, weight_kg):
    height_in_metres = height_cm / 100
    bmi_value = weight_kg / (height_in_metres * height_in_metres)
    return round(bmi_value, 2)

# The schema — this is what Claude READS (a job description)
bmi_tool_definition = {
    "name": "calculate_bmi",
    "description": "Calculates Body Mass Index from height and weight.",
    "input_schema": {
        "type": "object",
        "properties": {
            "height_cm": {"type": "number", "description": "Height in centimetres, e.g. 170"},
            "weight_kg": {"type": "number", "description": "Weight in kilograms, e.g. 68"}
        },
        "required": ["height_cm", "weight_kg"]
    }
}

user_question = "What is the BMI of someone who is 170cm tall and weighs 68kg?"

print("--- Step 1: Send message + tool definitions to Claude ---")
first_response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=[bmi_tool_definition],
    messages=[{"role": "user", "content": user_question}]
)

print("--- Step 2: Claude replied with stop_reason:", first_response.stop_reason, "---")
# Find the tool_use block in Claude's reply
tool_use_block = None
for block in first_response.content:
    if block.type == "tool_use":
        tool_use_block = block
        print("Claude wants tool:", block.name, "with inputs:", block.input)

print()
print("--- Step 3: OUR code runs the function (Claude does not!) ---")
bmi_result = calculate_bmi(tool_use_block.input["height_cm"],
                           tool_use_block.input["weight_kg"])
print("Our function returned:", bmi_result)

print()
print("--- Step 4: Send the tool result back to Claude ---")
# The conversation so far must be replayed:
# [user question, Claude's tool request, our tool result]
messages_with_result = [
    {"role": "user", "content": user_question},
    {"role": "assistant", "content": first_response.content},   # Claude's own request
    {"role": "user", "content": [{
        "type": "tool_result",
        "tool_use_id": tool_use_block.id,      # must match Claude's request ID
        "content": str(bmi_result)             # the result, as text
    }]}
]

final_response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=[bmi_tool_definition],
    messages=messages_with_result
)

print()
print("--- Step 5: Claude's final answer ---")
for block in final_response.content:
    if block.type == "text":
        print(block.text)

--- Step 1: Send message + tool definitions to Claude ---
--- Step 2: Claude replied with stop_reason: tool_use ---
Claude wants tool: calculate_bmi with inputs: {'height_cm': 170, 'weight_kg': 68}

--- Step 3: OUR code runs the function (Claude does not!) ---
Our function returned: 23.53

--- Step 4: Send the tool result back to Claude ---

--- Step 5: Claude's final answer ---
The BMI of someone who is 170cm tall and weighs 68kg is **23.53**.

This falls within the normal weight range. According to standard BMI classifications:
- **Underweight**: BMI < 18.5
- **Normal weight**: BMI 18.5 - 24.9
- **Overweight**: BMI 25.0 - 29.9
- **Obese**: BMI ≥ 30.0

A BMI of 23.53 is considered healthy and within the normal weight range.


**Expected result:** you see all 5 steps print in order, ending with a friendly sentence like *"A person who is 170cm and 68kg has a BMI of 23.53, which is in the healthy range."*

**Try this:** change the question to a different height and weight, and re-run.

Two details beginners often miss:

1. **The `tool_use_id` must match.** Claude's request has an ID; your result must quote the same ID. That's how Claude pairs results with requests when there are several.
2. **You replay the conversation.** The second API call includes the original question AND Claude's own tool request. The API doesn't remember your last call — every call must carry the full story so far.

### 🔬 Lab 3 — The AI *reasons* about the tool result

**Objective:** see that Claude doesn't just repeat the tool's output — it adds its own thinking on top.

The question asks two things: convert 38.5°C to Fahrenheit (**tool**), and "is that a fever?" (**Claude's own knowledge**).


In [6]:
# ============================================================
# LAB 3: Temperature converter — Claude reasons about the result
# ============================================================

def convert_celsius_to_fahrenheit(celsius):
    return round((celsius * 9 / 5) + 32, 1)

temperature_tool = {
    "name": "convert_celsius_to_fahrenheit",
    "description": "Converts a temperature from Celsius to Fahrenheit.",
    "input_schema": {
        "type": "object",
        "properties": {
            "celsius": {"type": "number", "description": "Temperature in Celsius, e.g. 38.5"}
        },
        "required": ["celsius"]
    }
}

question = "My body temperature is 38.5°C. What is that in Fahrenheit? Is it a fever?"

# Steps 1-2: send, receive the tool request
first_response = client.messages.create(
    model=MODEL, max_tokens=500,
    tools=[temperature_tool],
    messages=[{"role": "user", "content": question}]
)
tool_use_block = None
for block in first_response.content:
    if block.type == "tool_use":
        tool_use_block = block

# Step 3: our code converts
fahrenheit = convert_celsius_to_fahrenheit(tool_use_block.input["celsius"])
print("Tool ran:", tool_use_block.input["celsius"], "°C →", fahrenheit, "°F")

# Steps 4-5: send result back, get the final reasoned answer
final_response = client.messages.create(
    model=MODEL, max_tokens=500,
    tools=[temperature_tool],
    messages=[
        {"role": "user", "content": question},
        {"role": "assistant", "content": first_response.content},
        {"role": "user", "content": [{
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": str(fahrenheit)
        }]}
    ]
)

print()
print("Claude's final answer:")
for block in final_response.content:
    if block.type == "text":
        print(block.text)

print()
print("👆 The tool gave the NUMBER. Claude added the INTERPRETATION (fever or not).")

Tool ran: 38.5 °C → 101.3 °F

Claude's final answer:
Your body temperature of 38.5°C is **101.3°F**.

**Yes, this is a fever.** A normal body temperature is typically around 37°C (98.6°F). Generally:
- **37°C (98.6°F)** - Normal
- **37.5-38°C (99.5-100.4°F)** - Low-grade fever
- **38-39°C (100.4-102.2°F)** - Moderate fever
- **Above 39°C (102.2°F)** - High fever

At 38.5°C (101.3°F), you have a **moderate fever**. While fevers are generally your body's way of fighting an infection, you should:
- Stay hydrated
- Rest
- Monitor your symptoms
- Consider contacting a healthcare provider if the fever persists, worsens, or is accompanied by other concerning symptoms

If you have any additional symptoms or concerns, it's best to consult with a doctor.

👆 The tool gave the NUMBER. Claude added the INTERPRETATION (fever or not).


**Expected result:** Claude reports 101.3°F *and* explains that yes, this counts as a fever. The tool computed; Claude interpreted. That division of labour — **tools for facts, AI for judgement** — is the design pattern behind every serious AI assistant.

### 🔬 Lab 4 — One question, TWO tool calls

**Objective:** handle a question that needs the same tool twice — comparing two people's BMI.

Claude can request **multiple tool calls in one reply**. Your code must loop over ALL the blocks and answer each one.


In [7]:
# ============================================================
# LAB 4: Multiple tool calls in a single response
# ============================================================
# (reuses calculate_bmi and bmi_tool_definition from Lab 2)

question = ("Compare the BMI of Person A (160cm, 55kg) and Person B (180cm, 90kg). "
            "Who is closer to the healthy range?")

# Steps 1-2: send, receive
first_response = client.messages.create(
    model=MODEL, max_tokens=1000,
    tools=[bmi_tool_definition],
    messages=[{"role": "user", "content": question}]
)
print("Stop reason:", first_response.stop_reason)

# Step 3: run EVERY tool call Claude requested
all_tool_results = []
for block in first_response.content:
    if block.type == "tool_use":
        result = calculate_bmi(block.input["height_cm"], block.input["weight_kg"])
        print("Ran calculate_bmi", block.input, "→", result)
        all_tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,     # each result matched to its own request
            "content": str(result)
        })

# Steps 4-5: send ALL results back in ONE message
final_response = client.messages.create(
    model=MODEL, max_tokens=1000,
    tools=[bmi_tool_definition],
    messages=[
        {"role": "user", "content": question},
        {"role": "assistant", "content": first_response.content},
        {"role": "user", "content": all_tool_results}
    ]
)

print()
print("Claude's final comparison:")
for block in final_response.content:
    if block.type == "text":
        print(block.text)

Stop reason: tool_use
Ran calculate_bmi {'height_cm': 160, 'weight_kg': 55} → 21.48
Ran calculate_bmi {'height_cm': 180, 'weight_kg': 90} → 27.78

Claude's final comparison:
Based on the BMI calculations:

**Person A (160cm, 55kg):** BMI = **21.48**
**Person B (180cm, 90kg):** BMI = **27.78**

**Healthy BMI Range:** 18.5 - 24.9

**Analysis:**
- **Person A** has a BMI of 21.48, which falls **within the healthy range** (normal weight)
- **Person B** has a BMI of 27.78, which is **above the healthy range** (overweight category, since BMI ≥ 25)

**Conclusion:** **Person A is significantly closer to the healthy range** - in fact, Person A is already within the healthy BMI range. Person B would need to lose weight to reach the healthy range (would need to get below 80.1 kg to bring their BMI under 25).


**Expected result:** two tool calls (one per person), then a final answer comparing BMI 21.48 vs 27.78.

### 🧠 Quick check — count the calls!

In Lab 4: how many times did your code call the **Claude API**? How many times did it run the **local `calculate_bmi` function**?

> **Answer:** Claude API: **2** (once to get the tool requests, once for the final answer). Local function: **2** (once per person). Note: two *tool calls* still only needed two *API calls*, because both results went back together.

### 💬 Remember This

> *"The tool call loop is: send → Claude requests → I execute → I return the result → Claude answers. If stop_reason is tool_use, Claude is waiting for me — not the other way around."*

### ⚠️ Don't mix these up

- ❌ **Wrong:** "One question = one tool call."
  ✅ **Right:** Claude may request several tool calls for one question — loop over all blocks.
- ❌ **Wrong:** "The API remembers my previous call."
  ✅ **Right:** Every call must resend the whole conversation, including Claude's own tool request.
- ❌ **Wrong:** "I send the tool result as a plain chat message."
  ✅ **Right:** It goes back as a special `tool_result` block, with the matching `tool_use_id`.

### 🎤 Interview questions

**Q (beginner): "Walk me through the tool call loop."**
> *Model answer:* "My code sends the user message plus tool schemas. If Claude replies with stop_reason `tool_use`, I read the tool name and inputs from the tool_use block, run my real function, and send back a tool_result block with the matching ID. Claude then produces the final natural-language answer. If it replies `end_turn`, no tool was needed."

**Q (intermediate): "Why does the tool_result need a tool_use_id?"**
> *Model answer:* "Claude can request multiple tools at once. The ID pairs each result with the exact request it answers, so nothing gets mixed up."


---
## Section 4 — A Toolbox of Tools (and One Loop to Rule Them All)

### 🧰 Offering Claude a toolbox

So far we handed Claude one tool at a time. Real applications offer a **set of tools**, and Claude picks the right one for each request. This is powerful because:

- You build each tool once.
- Claude figures out *which* tool fits *which* request.
- You never write "if the user asks about BMI, use this tool" — Claude does that routing.

Two small patterns make this clean:

| Pattern | What it is |
|---|---|
| **Tool registry** | A dictionary mapping tool *names* → Python *functions*. When Claude says "run `convert_currency`", we look up the function by name. |
| **Reusable loop** | One function that runs the whole 5-step loop for *any* message and *any* tools — so we never copy-paste loop code again. |

### 📝 Preview — what should happen

| Prompt | Expected tool |
|---|---|
| "A person is 165cm and 72kg. BMI?" | `calculate_bmi` |
| "How much is 5000 rupees in US dollars?" | `convert_currency` |
| "How many words in: The quick brown fox..." | `get_word_count` |
| "My BMI (80kg, 175cm) AND convert ₹85,000 to USD" | **two different tools** |
| "What are the health risks of BMI over 30?" | **no tool** — knowledge answer |


In [8]:
# ============================================================
# Three tools + a tool registry
# ============================================================

def calculate_bmi(height_cm, weight_kg):
    height_in_metres = height_cm / 100
    return round(weight_kg / (height_in_metres * height_in_metres), 2)

def convert_currency(amount, from_currency, to_currency):
    # Mock rates for teaching — a real app would call a live exchange-rate API
    mock_rates = {
        ("INR", "USD"): 0.012, ("USD", "INR"): 83.5,
        ("INR", "EUR"): 0.011, ("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09
    }
    key = (from_currency.upper(), to_currency.upper())
    if key in mock_rates:
        return round(amount * mock_rates[key], 2)
    return f"Rate not available for {from_currency} to {to_currency}"

def get_word_count(text):
    return len(text.split())

bmi_tool = {
    "name": "calculate_bmi",
    "description": "Calculates the Body Mass Index from height in cm and weight in kg.",
    "input_schema": {
        "type": "object",
        "properties": {
            "height_cm": {"type": "number", "description": "Height in centimetres"},
            "weight_kg": {"type": "number", "description": "Weight in kilograms"}
        },
        "required": ["height_cm", "weight_kg"]
    }
}

currency_tool = {
    "name": "convert_currency",
    "description": "Converts an amount from one currency to another. Supports INR, USD, EUR.",
    "input_schema": {
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "The amount to convert"},
            "from_currency": {"type": "string", "description": "Source currency code, e.g. INR"},
            "to_currency": {"type": "string", "description": "Target currency code, e.g. USD"}
        },
        "required": ["amount", "from_currency", "to_currency"]
    }
}

word_count_tool = {
    "name": "get_word_count",
    "description": "Counts the number of words in a given piece of text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "text": {"type": "string", "description": "The text to count words in"}
        },
        "required": ["text"]
    }
}

# The registry: tool NAME → Python FUNCTION
tool_registry = {
    "calculate_bmi": calculate_bmi,
    "convert_currency": convert_currency,
    "get_word_count": get_word_count
}

all_tools = [bmi_tool, currency_tool, word_count_tool]
print("✅ Three tools registered:", list(tool_registry.keys()))

✅ Three tools registered: ['calculate_bmi', 'convert_currency', 'get_word_count']


### 🔁 The reusable loop

One function that handles the full loop for any prompt. Two safety details worth noticing (both are real-world habits, not just style):

1. **A turn limit** (`for turn in range(5)`) — so a confused model can never loop forever and burn money.
2. We continue looping **only while** `stop_reason == "tool_use"` — anything else (done, hit the token limit, etc.) safely exits.


In [9]:
# ============================================================
# The reusable tool call loop
# ============================================================

def run_tool_call_loop(user_message, available_tools, registry, max_turns=5):
    print("User:", user_message)
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):            # safety: never loop forever
        response = client.messages.create(
            model=MODEL,
            max_tokens=1000,
            tools=available_tools,
            messages=messages
        )
        print(f"--- API call #{turn + 1} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason != "tool_use":
            # Claude is done — collect and return the text answer
            return "".join(b.text for b in response.content if b.type == "text")

        # Claude wants tools: run every requested call
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  Claude requested: {block.name}({block.input})")
                function_to_run = registry[block.name]        # look up by name
                output = function_to_run(**block.input)       # run it
                print(f"  Our function returned: {output}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(output)
                })

        # Add Claude's request + our results to the conversation, then loop
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    return "(Stopped: reached the turn limit)"

print("✅ Reusable loop ready.")

✅ Reusable loop ready.


In [10]:
# Test 1 — should route to calculate_bmi
answer = run_tool_call_loop(
    "A person is 165cm and weighs 72kg. What is their BMI?",
    all_tools, tool_registry)
print()
print("Final answer:", answer)

User: A person is 165cm and weighs 72kg. What is their BMI?
--- API call #1 | stop_reason: tool_use ---
  Claude requested: calculate_bmi({'height_cm': 165, 'weight_kg': 72})
  Our function returned: 26.45
--- API call #2 | stop_reason: end_turn ---

Final answer: The BMI of a person who is 165 cm tall and weighs 72 kg is **26.45**.

This falls into the **overweight** category according to standard BMI classifications:
- Underweight: BMI < 18.5
- Normal weight: BMI 18.5 - 24.9
- Overweight: BMI 25.0 - 29.9
- Obese: BMI ≥ 30.0


In [11]:
# Test 2 — should route to convert_currency
answer = run_tool_call_loop(
    "How much is 5000 Indian Rupees in US Dollars?",
    all_tools, tool_registry)
print()
print("Final answer:", answer)

User: How much is 5000 Indian Rupees in US Dollars?
--- API call #1 | stop_reason: tool_use ---
  Claude requested: convert_currency({'amount': 5000, 'from_currency': 'INR', 'to_currency': 'USD'})
  Our function returned: 60.0
--- API call #2 | stop_reason: end_turn ---

Final answer: 5000 Indian Rupees (INR) is equal to **60 US Dollars (USD)**.


In [12]:
# Test 3 — TWO different tools in one request
answer = run_tool_call_loop(
    "I weigh 80kg and am 175cm. Also convert my monthly salary of 85000 INR to USD. Give me both answers.",
    all_tools, tool_registry)
print()
print("Final answer:", answer)

User: I weigh 80kg and am 175cm. Also convert my monthly salary of 85000 INR to USD. Give me both answers.
--- API call #1 | stop_reason: tool_use ---
  Claude requested: calculate_bmi({'height_cm': 175, 'weight_kg': 80})
  Our function returned: 26.12
  Claude requested: convert_currency({'amount': 85000, 'from_currency': 'INR', 'to_currency': 'USD'})
  Our function returned: 1020.0
--- API call #2 | stop_reason: end_turn ---

Final answer: Here are your results:

**BMI:** Your Body Mass Index is **26.12**, which falls into the **overweight** category (BMI of 25-29.9 is considered overweight).

**Salary Conversion:** Your monthly salary of **85,000 INR** converts to approximately **1,020 USD**.


In [13]:
# Test 4 — NO tool needed: Claude answers from knowledge
answer = run_tool_call_loop(
    "What are the health risks of a BMI over 30?",
    all_tools, tool_registry)
print()
print("Final answer:", answer)

User: What are the health risks of a BMI over 30?
--- API call #1 | stop_reason: end_turn ---

Final answer: A BMI over 30 is classified as **obese**, and it's associated with numerous significant health risks:

## Major Health Risks:

**Cardiovascular Diseases**
- Increased risk of heart disease and stroke
- High blood pressure (hypertension)
- Elevated cholesterol levels
- Atherosclerosis (hardening of arteries)

**Metabolic Disorders**
- Type 2 diabetes
- Insulin resistance
- Metabolic syndrome

**Respiratory Problems**
- Sleep apnea
- Asthma
- Obesity hypoventilation syndrome

**Joint and Mobility Issues**
- Osteoarthritis, especially in knees, hips, and lower back
- Increased joint stress and inflammation
- Reduced mobility and physical function

**Cancer Risk**
- Higher risk of certain cancers including:
  - Breast cancer (postmenopausal women)
  - Colon cancer
  - Endometrial cancer
  - Prostate cancer

**Mental Health**
- Depression and anxiety
- Low self-esteem
- Social stigma

**Expected results:** Test 1 uses `calculate_bmi`, Test 2 uses `convert_currency`, Test 3 uses **both** tools, and Test 4 uses **no tool at all** — one API call, straight to `end_turn`. Having tools available never *forces* Claude to use them.

**Try this:** ask *"How many words are in: The quick brown fox jumps over the lazy dog"* — which tool fires?

### 💬 Remember This

> *"A tool registry maps names to functions; one generic loop then serves every tool you will ever add. Adding a capability to your assistant becomes: write a function, write its description, add one line to the registry."*

### 🧠 Quick check — design your own toolbox!

You're building a **fitness app**. Invent 3 tools. For each: name, one-sentence purpose, and a user prompt that would trigger it.

> **Example answer:**
>
> | Tool | Purpose | Trigger prompt |
> |---|---|---|
> | `calculate_calories_burned` | Estimates calories burned in a workout | "I ran 5km in 30 min — calories?" |
> | `get_step_count` | Reads today's steps from the fitness tracker | "How many steps have I done today?" |
> | `log_water_intake` | Records glasses of water drunk | "Log 2 glasses of water" |

### 🎤 Interview question

**Q (intermediate): "You have 15 tools and Claude keeps picking the wrong one. What do you fix first?"**
> *Model answer:* "The tool descriptions. Claude routes using the `description` field, so I'd make each one state clearly what the tool does and when to use it, and make similar tools clearly distinct. I'd also check for overlapping tools that should be merged."


---
## Section 5 — Claude's Own Built-In Tools (Web Search, Files, Code Execution)

### 💡 The big reveal

Here's a secret hiding in plain sight: **the Claude app (claude.ai) is itself a tool-using assistant** — built on the exact loop you just coded.

When you chat with Claude on claude.ai and it searches the web, reads your uploaded PDF, or runs Python to analyse your CSV — that is tool use. Same idea, same loop. Someone at Anthropic defined those tools; Claude decides when to call them.

| What you see in the Claude app | The tool behind it | What actually happens |
|---|---|---|
| 🔎 Claude searches the web | Web search tool | Claude writes a search query → a search engine runs it → results come back → Claude answers with sources |
| 📎 You upload a file, Claude reads it | File tools | The file is stored; Claude calls a "read file" tool to pull its content in |
| 🧮 Claude analyses data / makes charts | Code execution tool | Claude *writes Python code* → the code runs in a secure sandbox → Claude reads the output and explains it |

Look at that last row again. In code execution, **the tool's input is code that Claude wrote itself**. The loop is still: AI → tool → result → AI continues.

### 🖥️ Client tools vs server tools

There are two kinds of tools, and now you know both:

| | **Client tools** (Sections 1–4) | **Server tools** (this section) |
|---|---|---|
| Who writes the function? | You | Anthropic |
| Who runs it? | **Your code** | **Anthropic's servers** |
| The loop | You implement it (5 steps) | Happens invisibly inside one API call |
| Examples | `calculate_bmi`, your database lookup | Web search, code execution |

With a server tool you just add it to the `tools` list — no function, no registry, no loop code. The search runs on Anthropic's side and the answer comes back finished, with citations.

### 🔬 Lab 5 — Give Claude real web search (one line!)

**Objective:** ask a question about *today* — impossible from training data — and let the server-side web search tool answer it.

> ⚠️ Web search must be enabled for your API organization (it is by default; an admin can turn it off in the Console). 💰 It costs $10 per 1,000 searches — so this cell costs about **1 cent**.


In [14]:
# ============================================================
# LAB 5: The web search SERVER tool — Anthropic runs it for you
# ============================================================

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[{
        "type": "web_search_20250305",   # a ready-made tool type — no schema, no function!
        "name": "web_search",
        "max_uses": 1                    # allow at most 1 search (cost control)
    }],
    messages=[{"role": "user", "content": "What is the current price of gold per gram in India? Answer briefly."}]
)

# Watch the loop that happened INSIDE this single API call:
print("Blocks in the response:")
for block in response.content:
    print("  -", block.type)

print()
print("Final answer:")
for block in response.content:
    if block.type == "text":
        print(block.text, end="")

Blocks in the response:
  - server_tool_use
  - web_search_tool_result
  - text

Final answer:
Today's gold price in India stands at ₹14,433 per gram for 24 karat gold, ₹13,230 per gram for 22 karat gold, and ₹10,825 per gram for 18 karat gold.

**Expected result:** the block list shows the whole loop that ran server-side — `text` (Claude deciding to search), `server_tool_use` (the query it wrote), `web_search_tool_result` (what came back), then `text` (the cited answer). You wrote **zero** loop code: Anthropic ran the tool and fed the result back to Claude automatically.

**Try this:** change the question to "What is 2 + 2?" — Claude should answer **without searching** (no `server_tool_use` block). The deciding step still works exactly as in Section 2.

### 📎 And file upload? Code execution?

Both follow the same pattern, run on Anthropic's side:

- **Files:** the API has a Files API — upload a document once, then reference it in messages. In the Claude app, the upload button is a friendly face on this.
- **Code execution:** add the code execution tool and Claude can write and run Python in a secure sandbox (no internet inside) — this is how Claude analyses spreadsheets and draws charts. Claude writes code → sandbox runs it → Claude reads the output → answers.

You don't need to memorise their setup today. You need the *idea*: **every "superpower" you've seen Claude use is a tool, running through the same loop you built by hand today.**

### 💬 Remember This

> *"Client tools: my code runs the function. Server tools: Anthropic runs it for me inside one API call. Web search, code execution, and file reading are server-side tools — the same loop, just hosted."*

### ⚠️ Don't mix these up

- ❌ **Wrong:** "Web search means the model's training data is live."
  ✅ **Right:** The model is still frozen; a *tool* fetches live data at answer time.
- ❌ **Wrong:** "For server tools I must write the function and the loop."
  ✅ **Right:** You only add the tool type to the `tools` list. Anthropic hosts the function and the loop.
- ❌ **Wrong:** "Claude runs code on my laptop when it uses code execution."
  ✅ **Right:** The code runs in a secure sandbox on Anthropic's servers, isolated from your machine.

### 🎤 Interview question

**Q: "What's the difference between client tools and server tools in the Claude API?"**
> *Model answer:* "Client tools are functions I define and execute — Claude sends a tool_use request, my code runs it and returns a tool_result. Server tools like web search and code execution are defined and executed by Anthropic — I just include the tool type in my request, and the search or execution happens server-side within the same API call, results and citations included."

### 🧠 Quick check

You ask claude.ai: *"Summarise this PDF I just uploaded."* Which built-in tool concept is at work, and who runs it?

> **Answer:** File tools — the file was stored on Anthropic's side and Claude pulls its content in via a tool. Anthropic runs it (server-side), not your code.


---
## Section 6 — The Architecture: Where Tools Sit in a Real System

### 🏗️ The picture to keep in your head

```
                                 ┌────────────────────────────┐
   ┌──────┐   1. question       │        YOUR APP            │
   │ USER │ ───────────────────▶│  (owns the loop + tools)   │
   └──────┘                     └─────────┬──────────────────┘
      ▲                                   │ 2. message + tool schemas
      │                                   ▼
      │                          ┌───────────────┐
      │ 6. final answer          │  CLAUDE API   │  "the brain"
      │                          │  (decides)    │
      │                          └──────┬────────┘
      │                                 │ 3. tool_use request
      │                                 ▼
   ┌──┴───────────────────────────────────────────┐
   │   4. YOUR APP runs the tool:                 │
   │   • a Python function     • a database query │
   │   • a payment API         • a web search     │
   │   5. ...and sends the result back to Claude  │
   └──────────────────────────────────────────────┘
```

**Who does what:**

| Component | Responsibility |
|---|---|
| Your app | Owns the loop, runs the tools, holds the API key, logs everything |
| Claude API | Decides which tool, extracts inputs, writes the final answer |
| Tools | Do one job each, reliably: fetch, calculate, act |

### 🏦 A real enterprise example

A bank's customer assistant might have these tools: `get_balance(account_id)`, `get_recent_transactions(account_id)`, `block_card(card_id)`, `find_nearest_branch(city)`. The customer types plain English; Claude routes to the right tool; the bank's systems do the real work. The AI never touches the database directly — it can only go through the tools the bank chose to expose. **That boundary is the security model.**

### ⚠️ Where it breaks (failure points to design for)

1. **The tool fails** (database down, API timeout). Don't crash — return an error message *as the tool result*. Claude will apologise and explain gracefully.
2. **Claude picks the wrong tool or wrong inputs.** Fix descriptions; validate inputs in your function before acting.
3. **Endless looping.** Always cap turns (our `max_turns=5`).
4. **Dangerous tools.** `block_card` is fine to automate; `transfer_money` should require a human "confirm? y/n" step before your code runs it. Rule of thumb: **the AI proposes, your code disposes.**

### 🛡️ Production notes (the short list that matters)

- **Security:** the model only gets the tools you register — keep risky actions behind human approval. Never put secrets in tool descriptions.
- **Cost:** every loop turn is an API call with the whole conversation resent. More tools + longer chats = more tokens. Haiku for routing keeps this cheap.
- **Observability:** log every tool call (name, inputs, output, time). When something goes wrong, this log is how you find out what the AI actually did.


---
## 🛠️ Mini Project — "WhereIsMyOrder" Customer Support Assistant

### 💼 The business case

You work for an online store. Support staff spend hours a day answering three questions: *Where is my order? Can I return this? How much is shipping to my city?* Your job: build an AI assistant that answers all three automatically — using tools, because the truth lives in company data, not in Claude's memory.

### 🏗️ Architecture

```
Customer ──▶ run_tool_call_loop ──▶ Claude (decides)
                    │                       │
                    ▼ runs tools            ▼ picks from
        ┌───────────────────────┬────────────────────────┐
        │ get_order_status      │  "orders database" (a dict)
        │ check_return_window   │  order date + return policy
        │ get_shipping_cost     │  city → price table
        └───────────────────────┴────────────────────────┘
```

### 📋 Build steps

1. Run the starter cell below — `get_order_status` is fully built for you.
2. Build tool 2: `check_return_window(order_id)` — returns whether the order is still within 10 days of the order date.
3. Build tool 3: `get_shipping_cost(city)` — look up a price from the dict.
4. Register all three tools and test the prompts at the bottom.

### ✅ Definition of done

- "Where is order ORD123?" → correct status from the data, in a friendly sentence.
- "Can I still return ORD124?" → correct yes/no with the reason.
- "What does shipping to Chennai cost?" → correct price.
- "What's your name?" → answered with **no** tool call (check the printout!).


In [15]:
# ============================================================
# MINI PROJECT starter — finish tools 2 and 3 yourself!
# ============================================================
import datetime

# --- The "company database" (mock data) ---
ORDERS = {
    "ORD123": {"item": "Blue running shoes", "status": "Shipped - arriving July 14", "order_date": "2026-07-08"},
    "ORD124": {"item": "Coffee maker",       "status": "Delivered on July 2",       "order_date": "2026-06-25"},
}
SHIPPING = {"chennai": 40, "mumbai": 60, "delhi": 60, "bangalore": 50}

# --- Tool 1: DONE for you ---
def get_order_status(order_id):
    order = ORDERS.get(order_id.upper())
    if order is None:
        return f"No order found with id {order_id}"
    return f"{order['item']}: {order['status']}"

order_status_tool = {
    "name": "get_order_status",
    "description": "Looks up the current status of a customer's order by its order id, e.g. ORD123.",
    "input_schema": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "The order id, e.g. ORD123"}},
        "required": ["order_id"]
    }
}

# --- Tool 2: YOUR TURN ---
# Return "Yes, N days left" if today is within 10 days of order_date, else "No, window closed".
def check_return_window(order_id):
    # TODO: look up ORDERS[order_id]["order_date"], compare with datetime.date.today()
    return "TODO"

# TODO: write return_window_tool schema (copy the shape of order_status_tool)

# --- Tool 3: YOUR TURN ---
def get_shipping_cost(city):
    # TODO: look up SHIPPING (hint: city.lower()), return the price or a polite "we don't ship there yet"
    return "TODO"

# TODO: write shipping_cost_tool schema

# --- Register + test (uncomment as you finish each tool) ---
project_tools = [order_status_tool]                 # add your two schemas here
project_registry = {"get_order_status": get_order_status}   # and your two functions here

print(run_tool_call_loop("Where is my order ORD123?", project_tools, project_registry))
# print(run_tool_call_loop("Can I still return ORD124?", project_tools, project_registry))
# print(run_tool_call_loop("How much is shipping to Chennai?", project_tools, project_registry))
# print(run_tool_call_loop("What's your name?", project_tools, project_registry))

User: Where is my order ORD123?
--- API call #1 | stop_reason: tool_use ---
  Claude requested: get_order_status({'order_id': 'ORD123'})
  Our function returned: Blue running shoes: Shipped - arriving July 14
--- API call #2 | stop_reason: end_turn ---
Your order ORD123 has been **shipped**! 

**Details:**
- **Item:** Blue running shoes
- **Status:** Shipped
- **Estimated Arrival:** July 14

Your package is on its way and should arrive by July 14. You should receive tracking information if you haven't already, so you can monitor its progress.


---
## 🎤 Explain It Out Loud — Your 60-Second Script

The goal of today was that **you can explain tool use clearly to anyone**. Here is your script. Read it aloud twice. Then close the notebook and say it from memory.

> *"An LLM on its own can only write text — its knowledge is frozen at training time, and it can't take actions. **Tool use** fixes that. We describe functions to the AI — each tool is real code plus a short description the AI reads. When a question needs live data, an exact calculation, or an action, the AI doesn't answer directly. Instead it returns a **tool_use request**: 'run this function, with these inputs' — inputs it extracted from plain English by itself. **The AI never runs the code — my code does.** I run the function, send the result back, and the AI continues, turning the raw result into a helpful answer. That's the loop: **AI → tool → result → AI continues.** It repeats until no more tools are needed. This is exactly how Claude's own web search, file reading, and code execution work — those are just tools hosted on Anthropic's servers. And it's the foundation of every AI agent."*

If you can say that — congratulations, you understand tool use better than most people using AI today.


---
## 📋 Session Summary

An LLM alone can only generate text from frozen knowledge. Tool use lets it *request* real actions: we describe functions (name + description + inputs), Claude decides when one is needed and returns a `tool_use` request with the inputs filled in, our code runs the real function and returns the result, and Claude continues with a final answer. One registry plus one loop scales this to any number of tools. The same mechanism, hosted server-side by Anthropic, powers Claude's own web search, file reading, and code execution.

---

## ✅ What You Learned Today

You can now:

- [ ] Explain **why** LLMs need tools (frozen knowledge, unreliable exact math, no actions)
- [ ] Define a tool as **function + schema**, and explain who reads vs runs each part
- [ ] Read `stop_reason` and explain the difference between `end_turn` and `tool_use`
- [ ] Implement the **5-step tool call loop** with the real Claude API
- [ ] Handle **multiple tool calls** in one response, matched by `tool_use_id`
- [ ] Build a **tool registry** and a **reusable loop** with a turn limit
- [ ] Use a **server tool** (web search) and explain client vs server tools
- [ ] Explain how claude.ai's file upload, web search, and code execution are tool use
- [ ] Deliver the 60-second explanation from memory


---
## 🗂️ AI Architect Cheat Sheet — Tool Use

**Definitions**

| Term | Meaning |
|---|---|
| Tool use / function calling | LLM returns a structured request for your code to run a function |
| Tool schema | Dict with `name`, `description`, `input_schema` — what Claude reads |
| `tool_use` block | Claude's request: tool name + inputs + id |
| `tool_result` block | Your reply: the output, tagged with the matching `tool_use_id` |
| Tool registry | Dict mapping tool names → functions |
| Client tool | You define it, you run it |
| Server tool | Anthropic defines and runs it (web search, code execution) |

**The loop**

```
send (message + tools) → stop_reason == "tool_use"?
   no  → done, read the text
   yes → run each tool_use block → append assistant content +
         tool_results → send again → repeat (with a turn limit!)
```

**Code quick-reference**

```python
tools = [{"name": "...", "description": "...",
          "input_schema": {"type": "object", "properties": {...}, "required": [...]}}]

r = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)

r.stop_reason          # "end_turn" or "tool_use"
block.type             # "text" or "tool_use"
block.name, block.input, block.id       # the request
{"type": "tool_result", "tool_use_id": block.id, "content": str(out)}

# Server tool — no function, no loop needed:
tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 1}]
```

**Decision table — does this need a tool?**

| Need | Tool? |
|---|---|
| Stable general knowledge | No |
| Live / current data | Yes (e.g. web search, an API) |
| Exact calculation that must be right | Yes |
| Action (email, DB write, booking) | Yes — with human approval if risky |


---
## ⏱️ 5-Minute Revision Guide

1. **Why tools?** LLM knowledge is frozen; it can't act; its math is "usually right" (not enough).
2. **A tool = function + description.** Claude reads the description. Your code runs the function.
3. **The decision:** Claude compares the question against the tool descriptions. Good descriptions = good routing.
4. **The loop:** send → `tool_use`? → run it → send `tool_result` (matching id) → Claude answers. Repeat while `stop_reason == "tool_use"`, with a turn cap.
5. **Many tools:** registry (name → function) + one reusable loop.
6. **Two kinds:** client tools (you run) vs server tools (Anthropic runs: web search, code execution, files).
7. **The sentence to never forget:** *the AI proposes, your code disposes.*

---

## 🎯 Interview Preparation Notes

**Q1. What is tool use?**
> The LLM, instead of answering directly, returns a structured request for my code to run a specific function with specific inputs; I run it, return the result, and the LLM writes the final answer.

**Q2. How does the model decide to call a tool?**
> It reads the user's request and the available tool descriptions, and judges whether it can answer from training or needs a tool. The description field is what steers this — routing quality is mostly description quality.

**Q3. Walk me through the loop.**
> Send message + schemas → if `stop_reason` is `tool_use`, read name/inputs/id from each tool_use block, execute, send back tool_result blocks with matching ids, and call again with the full conversation → repeat until `end_turn`.

**Q4. Does the model execute the function?**
> Never. It only requests. Execution, validation, and permissions live entirely in my code — that's the security boundary.

**Q5. Client vs server tools?**
> Client tools: I define and run them. Server tools (web search, code execution): Anthropic hosts and runs them; I just include the tool type; results and citations come back in the same API call.

**Q6 (architecture). Design an AI assistant that can issue refunds. What do you watch for?**
> Tools: `lookup_order`, `check_refund_eligibility`, `issue_refund`. The first two can auto-run; `issue_refund` needs human confirmation and strict input validation. Cap loop turns, log every tool call, return tool errors as tool_results so the model degrades gracefully, and keep credentials in my code — never in prompts.

**Q7 (architecture). One question triggers 3 tool calls — how many API calls?**
> Possibly just 2: one that returns all three requests (if made in parallel), one after I return all three results together. If the model chains them sequentially (needs one result to make the next call), more turns.


---
## 📝 Assignment

**Beginner** — Write, from memory, the 5 steps of the tool call loop. Then add a `reverse_text(text)` tool to the Section 4 toolbox and trigger it.

**Intermediate** — Build a `get_day_of_week(date_string)` tool (use Python's `datetime`). Test: "What day of the week is 2026-08-15?" Confirm the tool fires and the answer is right.

**Advanced** — Make `convert_currency` fail on purpose (raise an exception for unknown currencies). Update `run_tool_call_loop` to catch the error and return the error text as the tool_result. Watch Claude apologise gracefully instead of your program crashing.

**Project** — Finish the WhereIsMyOrder assistant (all 3 tools + all 4 test prompts pass). Then add a 4th tool of your own design and demo it to the class.

---

## 🧪 Assessment

### Part A — Multiple choice (10)

**1.** A tool, as given to Claude, consists of:
a) Python code Claude executes  b) a name, description, and input schema  c) a URL  d) a trained sub-model

**2.** Claude signals it wants a tool by:
a) `stop_reason == "end_turn"`  b) an email  c) `stop_reason == "tool_use"` and a tool_use block  d) running the function

**3.** Who actually executes a *client* tool's function?
a) Claude  b) Anthropic's servers  c) your code  d) the user

**4.** The `tool_use_id` exists so that:
a) billing works  b) each result can be matched to its exact request  c) tools stay secret  d) the loop ends

**5.** You give Claude 3 tools and ask "Explain what BMI means." What happens?
a) Claude uses calculate_bmi  b) Claude answers directly, no tool  c) error  d) Claude asks permission

**6.** After running a tool, you send the result back as:
a) a plain text message  b) a `tool_result` content block from the user role  c) a system prompt  d) a new tool schema

**7.** In one response Claude requests the same tool twice. Your code should:
a) run only the first  b) error out  c) run both and return both results with their ids  d) ask Claude to choose

**8.** Claude keeps picking the wrong tool. Best first fix:
a) bigger model  b) more max_tokens  c) rewrite the tool descriptions  d) remove all tools

**9.** Web search on the Claude API is a:
a) client tool you must implement  b) server tool Anthropic executes  c) separate product  d) training-data refresh

**10.** Why cap the loop with `max_turns`?
a) the API requires it  b) so a confused model can't loop forever and burn cost  c) tools break after 5 uses  d) Claude forgets otherwise

### Part B — Short answer (5)

**11.** Why can't an LLM know today's date without a tool?
**12.** What are the two parts of a tool, and who "consumes" each part?
**13.** Your messages list for the second API call contains three entries — name them in order.
**14.** Give one example of a tool that should require human approval before executing, and say why.
**15.** How do claude.ai's file upload and code execution features relate to what you built today?

### Part C — Scenarios (3)

**16.** A user asks your travel assistant: "Book me the cheapest flight to Delhi tomorrow." You have `search_flights` and `book_flight` tools. Describe the full sequence of API calls and tool calls, and where you'd insert a safety step.

**17.** In production, your assistant sometimes returns "(Stopped: reached the turn limit)". What are two likely causes, and one fix for each?

**18.** Your CEO asks: "Is it safe to let the AI query our customer database?" Give a two-sentence answer using what you learned about the security boundary of tool use.


---
## 🔑 Answer Key

**Part A:** 1-b · 2-c · 3-c · 4-b · 5-b · 6-b · 7-c · 8-c · 9-b · 10-b

**Part B:**

**11.** Its knowledge is frozen at training time — it has no clock and no live data; only a tool can inject current information.
**12.** The **function** (real code — your computer runs it) and the **schema/description** (Claude reads it to decide when and how to call).
**13.** ① the original user message, ② Claude's assistant message containing the tool_use block, ③ a user message containing the tool_result block(s).
**14.** Anything irreversible or money-moving, e.g. `issue_refund` or `transfer_money` — a wrong or hallucinated input would cause real damage, so a human confirms first.
**15.** They are the same mechanism — tool use — but as *server tools*: Anthropic hosts the file-reading and code-running functions and runs the loop on their side; the app is a friendly interface over it.

**Part C (model answers):**

**16.** Call 1: send the request + both schemas → Claude requests `search_flights(destination="Delhi", date=tomorrow)`. Your code runs it, returns results. Call 2: Claude picks the cheapest and requests `book_flight(flight_id=...)`. **Safety step here:** before executing `book_flight`, your code shows the human "Book flight X for ₹Y — confirm?" Only after confirmation do you run it and send the result; call 3 returns Claude's final confirmation message.
**17.** Cause A: a tool keeps erroring so Claude keeps retrying → fix: return the error clearly in tool_result and instruct (in the system prompt) to stop retrying after one failure. Cause B: the task genuinely needs more turns than the cap → fix: raise `max_turns` and monitor cost.
**18.** "The AI never touches the database directly — it can only request the specific, read-only queries we expose as tools, and our code validates every input before running them. Risky operations simply aren't given to it as tools, and everything it does request is logged."

---

## 📚 Continue Learning

- Tool use overview: https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview
- Build a tool-using agent (official tutorial): https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent
- Web search tool: https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool
- Code execution tool: https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool
- Files API: https://platform.claude.com/docs/en/build-with-claude/files

**Next session preview:** now that Claude can call one tool at a time, what happens when we let it chain many tools toward a goal, deciding its own next step? That's an **AI agent** — see you there. 🚀

*Questions? Reach out via the WhatsApp group.*
